In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
#cargamos los datos
df_puerto = pd.read_csv('datosPuerto.csv')
df_puerto.head()

In [ ]:
#Filtramos las partidas arancelarias para que solo salgan los capitulos
df_puerto['capitulo'] = df_puerto['PARTIDA_ARANCELARIA'].astype(str).str[:2]

In [ ]:
#FIltramos para que solo sean importanciones y exportaciones a Paises bajos
df_puerto = df_puerto[((df_puerto['PAIS_ORIGEN'] == 'NL') & (df_puerto['PAIS_FINAL'] == 'ES')) | ((df_puerto['PAIS_ORIGEN'] == 'ES') & (df_puerto['PAIS_FINAL'] == 'NL'))]

In [ ]:
#Agrupamos los kilos por pais y capitulo, y los ordenamos de mayor a menor para ver los capitulos mas importantes
ranking_puerto = df_puerto.groupby(['PAIS_ORIGEN', 'capitulo'])['PESO_KILOS'].sum().reset_index()
ranking_puerto = ranking_puerto.sort_values(by='PESO_KILOS', ascending=False)

In [ ]:
ranking_puerto

In [ ]:
#codigos_arancelarios_capitulos.pdf:

#CAPÍTULO 27:	Combustibles minerales, aceites minerales y productos de su destilación, materias bituminosas, ceras minerales.
#CAPÍTULO 72:	Fundición, hierro y acero
#CAPÍTULO 25:	Sal, azufre, tierras y piedras, yesos, cales y cementos. 
#CAPITULO 29:   Productos químicos orgánicos.

In [ ]:
#Cargamos los datos para hacer la importancia del puerto de bilbao en españa pero solo con los capitulos importantes
euskadi = pd.read_excel("importaciones_exportaciones_euskadi.xlsx",sheet_name="Importaciones")
puerto = pd.read_csv("datosPuerto.csv", decimal=",")

In [ ]:
#Cambiamos la fecha a datetime y ponemos solo el año
puerto['FECHA'] = pd.to_datetime(puerto['FECHA'])
puerto['FECHA'] = puerto['FECHA'].dt.year

In [ ]:
#Filtramos para que el periodo de años sea del 2009 al 2019(es hasta 2019 poprque el dataset de euskadi solo llega hasta 2019) y remplazamos los valores faltantes por 0
euskadi.copy()
euskadi_filtrado = euskadi[(euskadi["País"]== "Países Bajos")]
columnas_años = ["2009", "2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019"]
euskadi_filtrado[columnas_años] = (euskadi_filtrado[columnas_años].replace("-", 0))
euskadi_filtrado[columnas_años] = euskadi_filtrado[columnas_años].astype(float)


In [ ]:
#Filtramos para que solo sea el puerto respecto a paises bajos
puerto_paisesbajos = puerto[(puerto["PAIS_FINAL"] == "NL") | (puerto["PAIS_ORIGEN"] == "NL")]

In [ ]:
#Filtrammos de nuevo para que solo salgan los capitulos
puerto_paisesbajos['capitulo'] = puerto_paisesbajos['PARTIDA_ARANCELARIA'].astype(str).str[:2]

In [ ]:
puerto_paisesbajos

In [ ]:
#Cambiamos el numero de capitulo a int
puerto_paisesbajos["capitulo"]=puerto_paisesbajos["capitulo"].astype(int)

In [ ]:
#Filtramos para que solo sean las exportaciones y importaciones de los capitulos mencionados
capitulos = [25, 27, 29, 72]
codigos = ["25", "27", "29", "72"]

euskadi_filtrado = euskadi_filtrado[euskadi_filtrado["Código_arancelario"].str.startswith(tuple(codigos))]

puerto_paisesbajos = puerto_paisesbajos[puerto_paisesbajos["capitulo"].isin(capitulos)]

In [ ]:
#Hacemos la suma de las toneladas por año y lo pasamos a kilos
toneladas = euskadi_filtrado[euskadi_filtrado["Unidades"] == "Toneladas"]

importaciones_por_año = toneladas[columnas_años].sum()

importaciones_por_año = importaciones_por_año*1000

print(importaciones_por_año)

In [ ]:
puerto_paisesbajos

In [ ]:
#FIltramos solo paises bajos y españa
imp_puerto = puerto_paisesbajos[(puerto_paisesbajos["PAIS_ORIGEN"] == "NL") & (puerto_paisesbajos["PAIS_FINAL"] == "ES")]

In [ ]:
#agrupamos los kilos pòr años
imp_puerto = imp_puerto.groupby("FECHA")["PESO_KILOS"].sum()

In [ ]:
#Filtramos solo de 2009 a 2019
imp_puerto=imp_puerto.loc[2009:2019]

In [ ]:
imp_puerto

In [ ]:
importaciones_por_año.astype(int)

In [ ]:
#Lo pasamos a una serie para que sea mas sencillo el calculo del porcentaje
imp_puerto=pd.Series({
    2009: 558548317,
    2010: 813118973,
    2011: 495420144,
    2012: 585389939,
    2013: 413594698,
    2014: 442602112,
    2015: 318268047,
    2016: 312975651,
    2017: 347727342,
    2018: 294974762,
    2019: 445085736
})
importaciones_por_año=pd.Series({
    2009: 487628000,
    2010: 692902000,
    2011: 455078000,
    2012: 397200000,
    2013: 187808000,
    2014: 251330000,
    2015: 409402000,
    2016: 385262000,
    2017: 386290000,
    2018: 273190000,
    2019: 364605000
})

In [ ]:
#Calculamos la importancia del puerto de bilbao respecto a euskadi(DA mas de 100 porque el dataset de euskadi tiene un error en la recogida de datos)
porcentaje_imp = (imp_puerto / importaciones_por_año) * 100
porcentaje_imp